# Phase 3 — Schema RAG (ChromaDB)

Goal: embed table + column metadata into ChromaDB so the agent retrieves
only the relevant schema context per question instead of dumping everything
into the prompt. Only needed once your dataset has many columns.

Run notebooks 01 and 02 first.

In [ ]:
import sqlite3
import chromadb
from chromadb.utils import embedding_functions

DB_PATH    = 'data.db'
TABLE      = 'data'
COLLECTION = 'schema_store'

## Step 1 — extract column metadata from SQLite

In [ ]:
conn    = sqlite3.connect(DB_PATH)
cols    = conn.execute(f'PRAGMA table_info({TABLE})').fetchall()
samples = conn.execute(f'SELECT * FROM {TABLE} LIMIT 5').fetchall()
conn.close()

col_names = [c[1] for c in cols]

# build one document per column — this is what gets embedded
documents = []
for i, c in enumerate(cols):
    col_name = c[1]
    col_type = c[2]
    sample_vals = [str(row[i]) for row in samples if row[i] is not None][:3]
    doc = f'Table: {TABLE} | Column: {col_name} | Type: {col_type} | Sample values: {', '.join(sample_vals)}'
    documents.append(doc)

print(f'Built {len(documents)} column documents')
for d in documents:
    print(d)

## Step 2 — embed and store in ChromaDB

In [ ]:
chroma  = chromadb.Client()
ef      = embedding_functions.DefaultEmbeddingFunction()  # local, no API key needed

# delete collection if it already exists (useful when re-running)
try:
    chroma.delete_collection(COLLECTION)
except:
    pass

collection = chroma.create_collection(COLLECTION, embedding_function=ef)

collection.add(
    documents=documents,
    ids=[f'col_{i}' for i in range(len(documents))],
)
print(f'Stored {collection.count()} embeddings')

## Step 3 — retrieve relevant columns for a question

In [ ]:
def get_relevant_schema(question: str, n_results: int = 5) -> str:
    results = collection.query(query_texts=[question], n_results=n_results)
    docs    = results['documents'][0]
    return f'Table: {TABLE}\nRelevant columns:\n' + '\n'.join(f'  - {d}' for d in docs)

# test retrieval
test_questions = [
    'What is the total revenue by region?',
    'Show me the correlation between quantity and price',
    'Which customer spent the most?',
]

for q in test_questions:
    print(f'Q: {q}')
    print(get_relevant_schema(q))
    print('-' * 50)

## Step 4 — confirm it helps vs full schema

In [ ]:
# full schema token count (rough)
conn       = sqlite3.connect(DB_PATH)
cols_raw   = conn.execute(f'PRAGMA table_info({TABLE})').fetchall()
conn.close()
full_schema = 'Table: ' + TABLE + '\n' + '\n'.join(f'  - {c[1]} ({c[2]})' for c in cols_raw)

print('Full schema chars :', len(full_schema))
print('RAG schema chars  :', len(get_relevant_schema('total revenue by region')))
print()
print('RAG pays off once you have 20+ columns — for small datasets, full schema is fine')

## Next steps

Once retrieval returns the right columns for your questions,
`get_relevant_schema()` replaces `get_schema()` in `backend/ingestion.py`
and the ChromaDB logic moves into `backend/schema_store.py`.

After this: phase 4 = wire FastAPI (`backend/main.py`) and run end-to-end via HTTP.